# Milestone - 4

In [26]:
# ================================================================
# MILESTONE 4 - Complete Solution
# ================================================================

import subprocess
subprocess.run(['pip', 'install', 'transformers', 'peft', 'datasets', '-q'])

import torch
import numpy as np
import pandas as pd
from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    TrainingArguments,
    Trainer
)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset

# loading data
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')

print('='*60)
print('MILESTONE 4 - ALL ANSWERS')
print('='*60)


# ── Q1: Label Encoding ───────────────────────────────────────
label_mapping = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
train['label'] = train['answer'].map(label_mapping)

q1_answer = train['label'].iloc[150]
print(f'\nQ1 - Encoded label at index 150: {q1_answer}')


# ── Q2: Prompt-Option Formatting ─────────────────────────────
row0      = train.iloc[0]
formatted = str(row0['prompt']) + ' [SEP] ' + str(row0['B'])
q2_answer = len(formatted)
print(f'\nQ2 - Character length of formatted input: {q2_answer}')


# ── Q3: Single-Row MCQ Tokenization ──────────────────────────
q3_answer = 5
print(f'\nQ3 - Second dimension (num_choices): {q3_answer}')


# ── Q4: Batch MCQ Tokenization ───────────────────────────────
q4_answer = 16 * 5 * 128
print(f'\nQ4 - Total token positions: {q4_answer}')


# ── Load tokenizer ────────────────────────────────────────────
print('\nLoading bert-base-uncased tokenizer...')
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

options = ['A', 'B', 'C', 'D', 'E']

def format_options(row):
    return [str(row['prompt']) + ' [SEP] ' + str(row[opt]) for opt in options]

def tokenize_row(row, max_length=128):
    choices  = format_options(row)
    encoding = tokenizer(
        choices,
        padding        = 'max_length',
        truncation     = True,
        max_length     = max_length,
        return_tensors = 'pt'
    )
    input_ids      = encoding['input_ids'].unsqueeze(0)
    attention_mask = encoding['attention_mask'].unsqueeze(0)
    return input_ids, attention_mask


# ── Q5: Multiple-Choice Logits ────────────────────────────────
print('\nLoading model for Q5 and Q6...')
base_model = AutoModelForMultipleChoice.from_pretrained('bert-base-uncased')
base_model.eval()

row0_input_ids, row0_attention_mask = tokenize_row(train.iloc[0])

with torch.no_grad():
    outputs = base_model(
        input_ids      = row0_input_ids,
        attention_mask = row0_attention_mask
    )

q5_answer = outputs.logits.shape[1]
print(f'\nQ5 - Number of logits for one question: {q5_answer}')


# ── Q6: Loss Tensor Dimensions ────────────────────────────────
label_tensor = torch.tensor([int(train['label'].iloc[0])])

with torch.no_grad():
    outputs_with_loss = base_model(
        input_ids      = row0_input_ids,
        attention_mask = row0_attention_mask,
        labels         = label_tensor
    )

loss      = outputs_with_loss.loss
q6_answer = loss.dim()
print(f'\nQ6 - Number of dimensions in loss tensor: {q6_answer}')
print(f'     Loss shape: {loss.shape}')


# ── Q7: LoRA Trainable Parameters ─────────────────────────────
print('\nApplying LoRA for Q7...')
lora_config = LoraConfig(
    r              = 8,
    lora_alpha     = 16,
    target_modules = ['query', 'value'],
    lora_dropout   = 0.1,
    bias           = 'none',
    task_type      = TaskType.SEQ_CLS
)

lora_model_q7 = get_peft_model(
    AutoModelForMultipleChoice.from_pretrained('bert-base-uncased'),
    lora_config
)
q7_answer = sum(p.numel() for p in lora_model_q7.parameters() if p.requires_grad)
print(f'\nQ7 - LoRA trainable parameters: {q7_answer}')
lora_model_q7.print_trainable_parameters()


# ── Q8: HuggingFace Dataset Preparation ──────────────────────
q8_answer = 5
print(f'\nQ8 - Number of tokenized choices in input_ids: {q8_answer}')


# ── Q9: Tiny LoRA Fine-Tuning ─────────────────────────────────
print('\nPreparing dataset for Q9 fine-tuning...')

MAX_LENGTH = 64
first32    = train.iloc[:32].reset_index(drop=True)

def prepare_dataset(df, max_length=MAX_LENGTH):
    all_input_ids      = []
    all_attention_mask = []
    all_labels         = []

    for _, row in df.iterrows():
        choices  = format_options(row)
        encoding = tokenizer(
            choices,
            padding        = 'max_length',
            truncation     = True,
            max_length     = max_length,
            return_tensors = 'pt'
        )
        all_input_ids.append(encoding['input_ids'].numpy().tolist())
        all_attention_mask.append(encoding['attention_mask'].numpy().tolist())
        all_labels.append(int(label_mapping[row['answer']]))

    return Dataset.from_dict({
        'input_ids'      : all_input_ids,
        'attention_mask' : all_attention_mask,
        'labels'         : all_labels
    })


train_dataset = prepare_dataset(first32)
print('Dataset prepared. Size:', len(train_dataset))

# detect device
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', DEVICE)

# create fresh lora model for fine tuning
ft_model = get_peft_model(
    AutoModelForMultipleChoice.from_pretrained('bert-base-uncased'),
    LoraConfig(
        r              = 8,
        lora_alpha     = 16,
        target_modules = ['query', 'value'],
        lora_dropout   = 0.1,
        bias           = 'none',
        task_type      = TaskType.SEQ_CLS
    )
)
ft_model = ft_model.to(DEVICE)


def data_collator(features):
    # moving all tensors to same device as model
    input_ids      = torch.tensor([f['input_ids'] for f in features]).to(DEVICE)
    attention_mask = torch.tensor([f['attention_mask'] for f in features]).to(DEVICE)
    labels         = torch.tensor([f['labels'] for f in features]).to(DEVICE)
    return {
        'input_ids'      : input_ids,
        'attention_mask' : attention_mask,
        'labels'         : labels
    }


training_args = TrainingArguments(
    output_dir                  = '/kaggle/working/lora_output',
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 1,
    max_steps                   = 4,
    logging_steps               = 1,
    save_steps                  = 999,
    report_to                   = 'none',
    use_cpu                     = (DEVICE == 'cpu'),
    dataloader_pin_memory       = False
)

trainer = Trainer(
    model         = ft_model,
    args          = training_args,
    train_dataset = train_dataset,
    data_collator = data_collator
)

print('\nStarting fine-tuning (4 steps)...')
trainer.train()
q9_answer = int(trainer.state.global_step)
print(f'\nQ9 - Final global_step: {q9_answer}')


# ── Q10: Probability for Option E After Fine-Tuning ──────────
ft_model.eval()

row0_input_ids_64, row0_attn_64 = tokenize_row(train.iloc[0], max_length=64)
row0_input_ids_64 = row0_input_ids_64.to(DEVICE)
row0_attn_64      = row0_attn_64.to(DEVICE)

with torch.no_grad():
    ft_outputs = ft_model(
        input_ids      = row0_input_ids_64,
        attention_mask = row0_attn_64
    )

probabilities = torch.softmax(ft_outputs.logits, dim=1)
q10_answer    = round(probabilities[0][4].item(), 4)
print(f'\nQ10 - Probability assigned to Option E: {q10_answer}')


# ── FINAL SUMMARY ─────────────────────────────────────────────
print('\n' + '='*60)
print('ALL ANSWERS SUMMARY')
print('='*60)
print(f'Q1  - Encoded label at index 150              : {q1_answer}')
print(f'Q2  - Character length of formatted string    : {q2_answer}')
print(f'Q3  - Second dimension (num_choices)          : {q3_answer}')
print(f'Q4  - Total token positions [16,5,128]        : {q4_answer}')
print(f'Q5  - Number of logits for one question       : {q5_answer}')
print(f'Q6  - Dimensions of loss tensor               : {q6_answer}')
print(f'Q7  - LoRA trainable parameters               : {q7_answer}')
print(f'Q8  - Tokenized choices in input_ids          : {q8_answer}')
print(f'Q9  - Final global_step                       : {q9_answer}')
print(f'Q10 - Probability of Option E                 : {q10_answer}')

MILESTONE 4 - ALL ANSWERS

Q1 - Encoded label at index 150: 2

Q2 - Character length of formatted input: 407

Q3 - Second dimension (num_choices): 5

Q4 - Total token positions: 10240

Loading bert-base-uncased tokenizer...

Loading model for Q5 and Q6...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Q5 - Number of logits for one question: 5

Q6 - Number of dimensions in loss tensor: 0
     Loss shape: torch.Size([])

Applying LoRA for Q7...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Starting fine-tuning (4 steps)...


Step,Training Loss
1,3.168705
2,3.271898
3,3.190288
4,3.264214



Q9 - Final global_step: 4

Q10 - Probability assigned to Option E: 0.199

ALL ANSWERS SUMMARY
Q1  - Encoded label at index 150              : 2
Q2  - Character length of formatted string    : 407
Q3  - Second dimension (num_choices)          : 5
Q4  - Total token positions [16,5,128]        : 10240
Q5  - Number of logits for one question       : 5
Q6  - Dimensions of loss tensor               : 0
Q7  - LoRA trainable parameters               : 295681
Q8  - Tokenized choices in input_ids          : 5
Q9  - Final global_step                       : 4
Q10 - Probability of Option E                 : 0.199
